<a href="https://colab.research.google.com/github/pramodhugargol-0014/deep_learning/blob/main/UCI_Bank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install ucimlrepo

In [85]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from scipy.sparse import hstack
import pandas as pd
# from sklearn.utils.class_weight import compute_class_weight
# import numpy as np

# Load data
bank = fetch_ucirepo(id=222)

X = pd.DataFrame(bank.data.features)
y = pd.DataFrame(bank.data.targets)

# Convert target to 0/1
y = y["y"].map({"no": 0, "yes": 1})

# Split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# weights = compute_class_weight(
#     class_weight="balanced",
#     classes=np.unique(y_train),
#     y=y_train
# )

# class_weights = {
#     0: weights[0],
#     1: weights[1]
# }

# Identify columns using training data
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

binary_cols = [
    col for col in categorical_cols
    if X_train[col].nunique(dropna=False) == 2
]

multi_class_cols = [
    col for col in categorical_cols
    if X_train[col].nunique(dropna=False) > 2
]

# Encoders
binary_encoder = OrdinalEncoder()
multi_encoder = OneHotEncoder(handle_unknown="ignore")
scaler = StandardScaler()

# Numeric
X_train_numeric = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test_numeric = scaler.transform(
    X_test[numeric_cols]
)

# Binary categorical
X_train_binary = binary_encoder.fit_transform(
    X_train[binary_cols]
)

X_test_binary = binary_encoder.transform(
    X_test[binary_cols]
)

# Multi-category categorical
X_train_multi = multi_encoder.fit_transform(
    X_train[multi_class_cols]
)

X_test_multi = multi_encoder.transform(
    X_test[multi_class_cols]
)

# Combine
X_train_processed = hstack([
    X_train_numeric,
    X_train_binary,
    X_train_multi
])

X_test_processed = hstack([
    X_test_numeric,
    X_test_binary,
    X_test_multi
])

In [86]:
import tensorflow as tf
from tensorflow import keras
from keras import layers

model = keras.Sequential([
    layers.Input(shape=(48,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(48, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 64)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 48)             │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,857 (30.69 KB)

 Trainable params: 7,857 (30.69 KB)

 Non-trainable params: 0 (0.00 B)

In [87]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [88]:
print(X_train_processed.dtype)
print(X_test_processed.dtype)
print(y_train.dtype)
print(y_test.dtype)

X_train_processed = X_train_processed.astype("float32")
X_test_processed = X_test_processed.astype("float32")

y_train = y_train.astype("float32")
y_test = y_test.astype("float32")

float64
float64
int64
int64


In [93]:
history = model.fit(
    X_train_processed,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test_processed, y_test),
    # class_weight=class_weights
)

Epoch 1/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.9218 - loss: 0.1571 - val_accuracy: 0.9028 - val_loss: 0.2462
Epoch 2/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.9253 - loss: 0.1513 - val_accuracy: 0.9043 - val_loss: 0.2538
Epoch 3/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9271 - loss: 0.1500 - val_accuracy: 0.9048 - val_loss: 0.2564
Epoch 4/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9278 - loss: 0.1482 - val_accuracy: 0.9022 - val_loss: 0.2546
Epoch 5/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9289 - loss: 0.1465 - val_accuracy: 0.9030 - val_loss: 0.2658
Epoch 6/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9302 - loss: 0.1439 - val_accuracy: 0.8973 - val_loss: 0.2659
Epoch 7/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9330 - loss: 0.1417 - val_accuracy: 0.9018 - val_loss: 0.2637
Epoch 8/20
1131/1131 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9338 - loss: 0.139

In [97]:
y_pred = model.predict(X_test_processed)
y_pred_class = (y_pred >= 0.3).astype(int)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_class))

283/283 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
              precision    recall  f1-score   support

         0.0       0.95      0.92      0.94      7985
         1.0       0.52      0.63      0.57      1058

    accuracy                           0.89      9043
   macro avg       0.73      0.77      0.75      9043
weighted avg       0.90      0.89      0.89      9043



In [95]:
import joblib
import tensorflow as tf
from tensorflow import keras


# ============================================================
# 1. SAVE THE TRAINED MODEL
# ============================================================

model.save("bank_marketing_model.keras")


# ============================================================
# 2. SAVE THE PREPROCESSING OBJECTS
# ============================================================

joblib.dump(binary_encoder, "binary_encoder.pkl")
joblib.dump(multi_encoder, "multi_encoder.pkl")
joblib.dump(scaler, "scaler.pkl")


# ============================================================
# 3. SAVE THE COLUMN INFORMATION
# ============================================================

joblib.dump(numeric_cols.tolist(), "numeric_cols.pkl")
joblib.dump(binary_cols, "binary_cols.pkl")
joblib.dump(multi_class_cols, "multi_class_cols.pkl")


print("Model and preprocessing objects saved.")

Model and preprocessing objects saved.
